<a href="https://colab.research.google.com/github/beingAnujChaudhary/LLM-Zoomcamp/blob/main/02-vector-search/01.vectorSearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

# Mount Google Drive (optional)
drive.mount('/content/drive')

# Clone your GitHub repository
!git clone https://github.com/beingAnujChaudhary/LLM-Zoomcamp.git

# Move into repository
%cd /content/LLM-Zoomcamp

# Move into Module 2 folder
%cd 02-vector-search

Mounted at /content/drive
Cloning into 'LLM-Zoomcamp'...
remote: Enumerating objects: 88, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 88 (delta 42), reused 63 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (88/88), 2.92 MiB | 6.46 MiB/s, done.
Resolving deltas: 100% (42/42), done.
/content/LLM-Zoomcamp
/content/LLM-Zoomcamp/02-vector-search


# 📘 The Vector Search

---

Search is fundamentally about connecting a user's intent to the right data. Historically, this meant looking for exact word matches. Today, modern applications use Artificial Intelligence to understand the meaning behind a query. This masterclass bridges the gap between traditional search methodologies and modern, AI-driven Vector Search.

## Part 1: The Evolution of Information Retrieval
Information Retrieval (IR) is the science of searching for documents, information within documents, and metadata about documents.
*   **Boolean Era (1960s):** Exact matching using AND/OR/NOT operators.
*   **Probabilistic/Lexical Era (1970s-2000s):** TF-IDF and BM25. Matching based on word frequency and statistical rarity.
*   **Semantic/Dense Era (2010s-Present):** Word2Vec, Transformers, and Vector Search. Matching based on geometric proximity in high-dimensional space.

---


## Part 2: Keyword Search & The Semantic Gap

### 4. TF-IDF (Term Frequency-Inverse Document Frequency)
TF-IDF penalizes words that appear too frequently across all documents (like "the" or "is") and boosts words that are frequent in a specific document but rare globally.
*   **Formula:** $TF\text{-}IDF(t, d) = TF(t, d) \times \log\left(\frac{N}{df(t)}\right)$
  *   $TF$: Frequency of term $t$ in document $d$.
  *   $N$: Total number of documents.
  *   $df(t)$: Number of documents containing term $t$.

### 5. BM25 (Best Matching 25)
BM25 is the industry-standard probabilistic refinement of TF-IDF used in Elasticsearch and Solr. It introduces **term frequency saturation** and **document length normalization**.
*   **Formula:** $Score(D,Q) = \sum_{i=1}^{n} IDF(q_i) \times \frac{f(q_i,D) \cdot (k_1 + 1)}{f(q_i,D) + k_1 \cdot (1 - b + b \cdot \frac{|D|}{avgdl})}$
  *   $k_1$: Saturation parameter (usually 1.2 - 2.0). Prevents a term appearing 100 times from scoring 100x higher than appearing 10 times.
  *   $b$: Length normalization (usually 0.75). Penalizes longer documents.

### 6-7. Why Keyword Search Fails (The Semantic Gap)
Keyword search suffers from the **Vocabulary Mismatch Problem**.
*   **Query:** *"Can I still join the course after the start date?"*
*   **Document:** *"Is it possible to enroll late?"*
*   **Lexical Overlap:** ~10% (only "the" or "I").
*   **Semantic Overlap:** 100%.
Keyword search fails here because it lacks context, synonym handling, and paraphrase understanding.

---


## Part 3: Semantic Search & Embeddings

### 8. Semantic Search
Semantic search overcomes the Vocabulary Mismatch Problem by matching ideas and meaning rather than exact characters. If a user searches for "feline", semantic search knows to retrieve documents containing the word "cat", even if "feline" is completely absent from the text.

Instead of matching discrete tokens, semantic search maps text into a continuous, dense mathematical space where **semantic meaning translates to geometric proximity**.

### 9. Embeddings (Word vs. Sentence)

An **Embedding** is a mathematical representation of text as a dense vector (an array of numbers or $N$ floating-point numbers) in a continuous, multi-dimensional space.

*   **Static Word Embeddings (Word2Vec, GloVe):** Map individual words to vectors. *Flaw:* The word "bank" gets the same vector whether you mean a river bank or a financial bank (Polysemy problem).
*   **Contextual Sentence Embeddings (BERT, Sentence-Transformers):** Use the **Transformer architecture** and **self-attention** to encode entire sentences. The vector for "judge" changes dynamically based on surrounding words (e.g., "court" vs "evaluate LLM").

---



## Part 4: Mathematical Foundations

### 10-11. Linear Algebra & Vector Space

A vector is an ordered array of numbers (e.g., `[0.12, -0.34, 0.56]`).

*   **Dimensionality:** The number of components in the array (e.g., standard models use 384, 768, or 1536 dimensions).
*   **Magnitude (Norm):** The physical length or magnitude of the vector.
*   **Direction:** The orientation in the multi-dimensional space.

When we process thousands of documents, each one becomes a single point in a high-dimensional **vector space**.

* **Semantic Proximity:** Because the neural network was trained on human language, vectors with similar meanings naturally land near each other geometrically.
* **Dense Vectors:** Unlike one-hot encoding, most values in these arrays are non-zero.

### 12. Similarity Metrics & The Normalization Trick
To measure how "close" two vectors are, we need a distance metric.

**1. Cosine Similarity:** Measures the *angle* ($\theta$) between vectors, ignoring magnitude.
$$ \cos(\theta) = \frac{A \cdot B}{||A||_2 \times ||B||_2} $$
*Range: [-1, 1]. (1 = identical, 0 = orthogonal, -1 = opposite).*

**2. Dot Product (Inner Product):** The algebraic sum of corresponding components.
$$ A \cdot B = \sum_{i=1}^{n} A_i B_i $$

**3. Euclidean Distance (L2):** Straight-line physical distance. Sensitive to magnitude.

> 💡 **CRUCIAL TECHNICALITY: The Normalization Trick**
> Calculating the denominator in Cosine Similarity ($||A||_2 \times ||B||_2$) is computationally expensive. If we **L2-normalize** our vectors during indexing (scaling them so their length $||A||_2 = 1$), the denominator becomes $1 \times 1 = 1$.
> **Therefore, for normalized vectors, Cosine Similarity is mathematically identical to the Dot Product.**
> Vector databases (like FAISS) use Dot Product indexes (`IndexFlatIP`) under the hood because it is significantly faster to compute.

---


## Part 5: The Vector Pipeline & ANN Algorithms

### 13. The Two-Phase Pipeline
1.  **Offline (Indexing):** Pass all documents through an Embedding Model to convert them into vectors and store them in a Vector Database.

Text $\rightarrow$ Chunking $\rightarrow$ Embedding Model $\rightarrow$ Vector Arrays $\rightarrow$ Vector Index.


2.  **Online (Querying):** When a user asks a question, pass the query through the exact same model. The system searches the database for document vectors closest to the query vector

Query $\rightarrow$ Embedding Model $\rightarrow$ Query Vector $\rightarrow$ ANN Search $\rightarrow$ Top-K Results.

### 14. Approximate Nearest Neighbors (ANN)
Brute-force search (calculating distance against every vector) is $O(N)$. For 10 million documents, this is too slow. Production systems use ANN to achieve $O(\log N)$ or $O(\sqrt{N})$ speed.
*   **HNSW (Hierarchical Navigable Small World):** Builds a multi-layered graph for rapid traversal. Top layers are sparse (allowing massive "jumps" across the space), bottom layers are dense (allowing precise local search). *Industry standard for high-recall, low-latency.*
*   **IVF (Inverted File Index):** Uses K-Means clustering to divide the space into "Voronoi cells". Search only occurs in the closest clusters.
*   **PQ (Product Quantization):** Compresses vectors by breaking them into sub-vectors and clustering them, drastically reducing RAM usage.

---


## Part 6: Vector Databases

* **Vector Extensions**: Tools like `pgvector` or `sqlite-vec` add vector math capabilities to existing relational databases, ensuring ACID transactions.

* **Native Vector DBs**: Platforms like `Pinecone, Milvus, Qdrant, and ChromaDB` are built from the ground up for massive-scale vector math, advanced filtering, and clustering.

| Tool | Architecture | Pros | Cons |
| :--- | :--- | :--- | :--- |
| **Minsearch** | In-Memory (Python) | Simple, zero-dependency, great for prototyping. | $O(N)$ brute force, lost on restart. |
| **SQLite-vec** | Relational Extension | ACID compliant, keeps metadata and vectors together. | Not optimized for massive scale. |
| **PGVector** | PostgreSQL Extension | Enterprise-ready, SQL ecosystem, highly scalable. | Requires Postgres expertise, heavy setup. |
| **Pinecone/Qdrant** | Native Vector DB | Built from scratch for vectors, managed, distributed. | Separate infrastructure stack, vendor lock-in. |

---


## Part 7: RAG Integration & Production Best Practices

Because Vector Search is bad at exact term matching (like specific IDs), modern Retrieval-Augmented Generation (RAG) systems use Hybrid Search.
This runs both a Keyword Search and a Vector Search simultaneously. Because BM25 scores and Cosine scores are on different mathematical scales, they are merged using Reciprocal Rank Fusion (RRF), which scores based on rankings rather than raw distances.

### 1. Document Chunking
Embedding models perform best on small, coherent blocks of text.
*   **Recursive Character Splitting:** Splits by paragraphs $\rightarrow$ sentences $\rightarrow$ words to preserve semantic units.
*   **Overlap:** Crucial! You must overlap chunks (e.g., 500 tokens with 50 tokens overlap) so context at the boundaries isn't severed.

### 2. Hybrid Search & Reciprocal Rank Fusion (RRF)
Vector search fails at exact keyword matching (e.g., searching for an error code like `ERR-404`). **Hybrid Search** runs BM25 and Vector search simultaneously.
Because BM25 scores and Cosine scores are on different scales, we cannot simply add them. We use **RRF**, which ignores raw scores and only looks at rank:
$$ RRF\_score(d) = \sum_{r \in R} \frac{1}{k + rank_r(d)} $$
*(where $k$ is a constant, usually 60, to prevent top ranks from dominating).*

### 3. Quantization
Storing millions of 384-dimensional Float32 vectors requires massive RAM. **Scalar Quantization** converts 32-bit floats to 8-bit integers, reducing memory footprint by 4x with only a ~1% drop in accuracy.

### 4. Evaluation Metrics
*   **Precision@K:** Out of the top K retrieved, how many are actually relevant?
*   **Recall@K:** Out of all relevant documents in the DB, how many were found in the top K?
*   **NDCG (Normalized Discounted Cumulative Gain):** Measures ranking quality (penalizes relevant documents appearing lower in the list).

### Production Best Practices
* **Start Simple**: Never start with vector search. Start with basic text search and upgrade to hybrid/vector once the operational complexity is justified.

* **Chunking**: LLMs have context limits. Overlap chunks (e.g., 500 tokens with 50 tokens overlap) so context isn't lost at the boundaries.

* **Quantization**: Compress high-dimensional vectors (e.g., 32-bit floats down to 8-bit integers) to reduce memory footprints by 4x to 32x with only a 1-2% drop in accuracy.
---


## Part 8: Interview Questions, Exercises & Cheat Sheet

### 🎤 Top 5 Interview Questions
1.  **Q: Why do production vector databases prefer Dot Product over Cosine Similarity?**
    *   *A:* Because vectors are L2-normalized during indexing. For normalized vectors, Dot Product is mathematically identical to Cosine Similarity but requires significantly fewer CPU/GPU cycles to compute.
2.  **Q: Explain how HNSW works.**
    *   *A:* HNSW builds a multi-layered graph. The top layers are sparse, allowing the search algorithm to quickly jump across distant regions of the vector space. As it descends to the bottom layers, the graph becomes dense, allowing for precise, localized neighbor traversal.
3.  **Q: What is the Curse of Dimensionality in the context of vector search?**
    *   *A:* As dimensions increase, the volume of the space increases exponentially, making the data sparse. In high dimensions, the difference in Euclidean distance between the "nearest" and "farthest" points approaches zero, making distance metrics less meaningful. This is why Cosine Similarity (angle) is preferred over Euclidean (distance) for high-dimensional embeddings.
4.  **Q: Why use Hybrid Search instead of just Vector Search?**
    *   *A:* Vector search is "black box" and struggles with exact matches (like IDs, SKUs, or specific error codes). Keyword search (BM25) excels at exact matches but fails on synonyms. Hybrid search combines both to cover all retrieval bases.
5.  **Q: What is Reciprocal Rank Fusion (RRF)?**
    *   *A:* RRF is an algorithm used to merge results from multiple search retrievers (like BM25 and Vector) that have incomparable score scales. It calculates a new score based purely on the document's rank in each list: $1 / (k + rank)$.

### 🛠️ Practical Exercises
1.  **Exercise 1:** Modify the  code below to use `IndexIVFFlat` in FAISS instead of `IndexFlatIP`. Measure the difference in search time for 10,000 random vectors.
2.  **Exercise 2:** Implement a basic RRF algorithm in Python that takes two lists of document IDs (one from BM25, one from Vector) and merges them.
3.  **Exercise 3:** Take a dataset of 100 FAQ questions. Embed them, and use UMAP to reduce the dimensions to 2D. Plot them using `matplotlib` to visually verify that similar questions cluster together.

### 📝 Quick Cheat Sheet
| Concept | Definition / Formula |
| :--- | :--- |
| **Embedding** | Dense array of floats representing semantic meaning. |
| **Cosine Similarity** | $\frac{A \cdot B}{||A|| ||B||}$. Measures angle. Range: [-1, 1]. |
| **Dot Product** | $A \cdot B$. Equals Cosine Sim if vectors are L2-normalized. |
| **HNSW** | Graph-based ANN. High memory, extremely fast, high recall. |
| **IVF** | Cluster-based ANN. Lower memory, requires training step. |
| **RRF** | $\sum \frac{1}{k + rank}$. Merges hybrid search results. |
| **Chunking** | Splitting text. Always use overlap (e.g., 10-20%) to preserve context. |

---


## Part 9: Code Implementation

### Cell 1: Install Dependencies


In [2]:
# Install required libraries
# sentence-transformers: For generating embeddings
# faiss-cpu: Facebook's library for highly efficient similarity search (ANN)
# scikit-learn: For L2 normalization and TF-IDF comparison
!pip install -q sentence-transformers faiss-cpu scikit-learn numpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 83.4 MB/s eta 0:00:00


### Cell 2: Setup Data and Generate Embeddings


In [3]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Load a lightweight, fast embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [16]:
# 2. Define a sample dataset (FAQ style)

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

raw_documents = []
url_prefix = "https://datatalks.club/faq"

print("Fetching documents from DataTalks.Club...")
for course in courses_raw:
    course_url = f"{url_prefix}{course['path']}"
    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()
    raw_documents.extend(course_data)

print(f"Total documents fetched: {len(raw_documents)}")


Fetching documents from DataTalks.Club...
Total documents fetched: 1350


In [17]:
# Prepare text for embeddings
# Embedding models need strings, not dictionaries.
# We combine metadata (course/section) with the Q&A to give the model full context.
def prepare_text(doc):
    text = f"Course: {doc.get('course', '')}\n"
    text += f"Section: {doc.get('section', '')}\n"
    text += f"Question: {doc.get('question', '')}\n"
    text += f"Answer: {doc.get('answer', '')}"
    return text

# Create a list of formatted strings
document_texts = [prepare_text(doc) for doc in raw_documents]

print("\nSample document text being sent to the embedding model:")
print("-" * 50)
print(document_texts[0][:400] + "...")


Sample document text being sent to the embedding model:
--------------------------------------------------
Course: data-engineering-zoomcamp
Section: General Course-Related Questions
Question: Course: When does the course start?
Answer: A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).

- Register via the link in the course repo before the cohort...


In [20]:
# 3. Generate raw embeddings
# show_progress_bar=True is highly recommended for datasets > 100 docs
print("\nGenerating embeddings for 1350 documents...")
raw_embeddings = model.encode(
    document_texts,
    convert_to_numpy=True,
    show_progress_bar=True
)



Generating embeddings for 1350 documents...


Batches:   0%|          | 0/43 [00:00<?, ?it/s]

In [21]:
print(f"\nRaw Embeddings Shape: {raw_embeddings.shape}")
# Output: (1350, 384)


Raw Embeddings Shape: (1350, 384)


In [22]:
# 4. L2 NORMALIZATION (The Crucial Technical Step)
# We scale the vectors so their length (magnitude) is exactly 1.
normalized_embeddings = normalize(raw_embeddings, norm='l2')


In [23]:
# Verify they are normalized (the L2 norm of each row should be 1.0)
print(f"L2 Norms after normalization: {np.linalg.norm(normalized_embeddings, axis=1)}")


L2 Norms after normalization: [1.        1.0000001 1.        ... 1.        1.0000001 1.0000001]


### Cell 3: The Semantic Gap (TF-IDF vs Vector Search)


In [24]:
# Let's test with a query that uses different words but means the same thing
query = "Can I join the course late?"

# --- KEYWORD SEARCH (TF-IDF) ---
vectorizer = TfidfVectorizer()
doc_tfidf = vectorizer.fit_transform(document_texts)
query_tfidf = vectorizer.transform([query])
tfidf_scores = cosine_similarity(query_tfidf, doc_tfidf)[0]

In [25]:
# --- VECTOR SEARCH (Semantic) ---
query_raw = model.encode([query], convert_to_numpy=True)
query_norm = normalize(query_raw, norm='l2')
vector_scores = np.dot(normalized_embeddings, query_norm.T).flatten()

In [26]:
# Get top 3 results for both
top_k = 3
tfidf_top_indices = np.argsort(tfidf_scores)[::-1][:top_k]
vector_top_indices = np.argsort(vector_scores)[::-1][:top_k]

print(f"\n--- Query: '{query}' ---")
print("\n🔴 Top 3 Keyword Search (TF-IDF) Results:")
for idx in tfidf_top_indices:
    print(f"Score: {tfidf_scores[idx]:.4f} | Q: {raw_documents[idx]['question'][:70]}...")

print("\n🟢 Top 3 Vector Search (Semantic) Results:")
for idx in vector_top_indices:
    print(f"Score: {vector_scores[idx]:.4f} | Q: {raw_documents[idx]['question'][:70]}...")

# Notice: Vector search easily finds documents about "enrolling late" or "starting after the date",
# even if the exact word "late" isn't in the document!


--- Query: 'Can I join the course late?' ---

🔴 Top 3 Keyword Search (TF-IDF) Results:
Score: 0.4227 | Q: Am I too late to join the course?...
Score: 0.3225 | Q: I may end up submitting the assignment late. Would it be evaluated?...
Score: 0.3029 | Q: Homework: Are late submissions of homework allowed?...

🟢 Top 3 Vector Search (Semantic) Results:
Score: 0.6818 | Q: Am I too late to join the course?...
Score: 0.6535 | Q: Course - Can I still join the course after the start date?...
Score: 0.6423 | Q: The course has already started. Can I still join it?...


### Cell 4: Mathematical Proof (`Dot Product == Cosine Similarity`)


In [28]:
# Let's mathematically prove the Normalization Trick.

# Method A: Scikit-Learn Cosine Similarity (Calculates angles, slower)
cos_scores = cosine_similarity(query_norm, normalized_embeddings)[0]

# Method B: Dot Product (Algebraic sum, much faster)
dot_scores = np.dot(normalized_embeddings, query_norm.T).flatten()

print("\n--- Mathematical Proof ---")
print("Are Cosine Similarity and Dot Product identical?", np.allclose(cos_scores, dot_scores))



--- Mathematical Proof ---
Are Cosine Similarity and Dot Product identical? True


In [29]:
# In production, we use ANN indexes.
dimension = normalized_embeddings.shape[1] # 384

# IndexFlatIP = Inner Product (Dot Product).
# Because our vectors are L2-normalized, Inner Product == Cosine Similarity.
index = faiss.IndexFlatIP(dimension)

# Add all 1350 vectors to the index
index.add(normalized_embeddings)

# Perform a search for the top 3 closest documents
k = 3
distances, indices = index.search(query_norm, k)

print(f"\n--- FAISS Top {k} Results (Using Inner Product) ---")
for i in range(k):
    doc_idx = indices[0][i]
    score = distances[0][i]
    doc = raw_documents[doc_idx] # Fetch the original dictionary

    print(f"Rank #{i+1} | Score: {score:.4f}")
    print(f"Course: {doc['course']}")
    print(f"Section: {doc['section']}")
    print(f"Question: {doc['question']}")
    print(f"Answer: {doc['answer'][:150]}...\n")
    print("-" * 60)


--- FAISS Top 3 Results (Using Inner Product) ---
Rank #1 | Score: 0.6818
Course: ai-dev-tools-zoomcamp
Section: General Course-Related Questions
Question: Am I too late to join the course?
Answer: No. You're only late for submitting the first couple of homeworks, and they aren't required for the certificate. To earn the certificate you need to p...

------------------------------------------------------------
Rank #2 | Score: 0.6535
Course: mlops-zoomcamp
Section: General Course-Related Questions
Question: Course - Can I still join the course after the start date?
Answer: Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.

Be aware,...

------------------------------------------------------------
Rank #3 | Score: 0.6423
Course: machine-learning-zoomcamp
Section: General Course-Related Questions
Question: The course has already started. Can I still join it?
Answer: Yes, you can. Even though you mi